In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint


In [2]:
TRAIN_DIR = r"C:\Users\HP\OneDrive\Desktop\DisaterAssesment\dataset\train_set"
TEST_DIR  = r"C:\Users\HP\OneDrive\Desktop\DisaterAssesment\dataset\test_set"


In [3]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(
    rescale=1./255
)

In [4]:
x_train = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(64, 64),
    batch_size=16,
    color_mode='rgb',
    class_mode='categorical'
)

x_test = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(64, 64),
    batch_size=16,
    color_mode='rgb',
    class_mode='categorical'
)

Found 742 images belonging to 4 classes.
Found 198 images belonging to 4 classes.


In [5]:
NUM_CLASSES = x_train.num_classes
print("Number of classes:", NUM_CLASSES)
print("Class labels:", x_train.class_indices)

Number of classes: 4
Class labels: {'Cyclone': 0, 'Earthquake': 1, 'Flood': 2, 'Wildfire': 3}


In [6]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    Flatten(),

    Dense(128, activation='relu'),
    Dropout(0.5),

    Dense(NUM_CLASSES, activation='softmax')
])

C:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 62, 62, 32)          │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 31, 31, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 29, 29, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 14, 14, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 12, 12, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 6, 6, 128)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 4608)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │         589,952 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 4)                   │             516 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 683,716 (2.61 MB)

 Trainable params: 683,716 (2.61 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "disaster_assessment_model.h5",
    monitor='val_accuracy',
    save_best_only=True
)

In [10]:
history = model.fit(
    x_train,
    validation_data=x_test,
    epochs=20
)


C:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 22s 435ms/step - accuracy: 0.3059 - loss: 1.3664 - val_accuracy: 0.3333 - val_loss: 1.3250
Epoch 2/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 11s 236ms/step - accuracy: 0.3935 - loss: 1.2944 - val_accuracy: 0.5000 - val_loss: 1.2683
Epoch 3/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 11s 245ms/step - accuracy: 0.5229 - loss: 1.1559 - val_accuracy: 0.5556 - val_loss: 1.1515
Epoch 4/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 11s 237ms/step - accuracy: 0.5216 - loss: 1.0634 - val_accuracy: 0.7172 - val_loss: 1.0000
Epoch 5/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 11s 236ms/step - accuracy: 0.5809 - loss: 0.9894 - val_accuracy: 0.6717 - val_loss: 1.0119
Epoch 6/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 11s 229ms/step - accuracy: 0.6509 - loss: 0.8873 - val_accuracy: 0.6263 - val_loss: 0.8911
Epoch 7/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 11s 236ms/step - accuracy: 0.6590 - loss: 0.8183 - val_accuracy: 0.7273 - val_loss: 0.7975
Epoch 8/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 24s 518ms/step - accuracy: 0.6752 - loss: 0.7929 - val_accu

In [11]:
loss, accuracy = model.evaluate(x_test)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 297ms/step - accuracy: 0.7475 - loss: 0.7090
Test Accuracy: 74.75%


In [12]:
model.save("final_disaster_assessment_model.keras")
print("Model saved successfully!")

Model saved successfully!
